# Catboost - Classificação Multi Label

In [144]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, f_oneway, ttest_ind
from colorama import Fore, Back, Style
import plotly.express as px
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, \
                            log_loss, roc_curve, roc_auc_score
import shap

## Ler dados

In [145]:
df_companies = pd.read_csv("./dataset/companies_profile.csv")
df_companies.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ID                         1000 non-null   int64  
 1   Nome_Empresa               1000 non-null   str    
 2   Receita_Anual              1000 non-null   int64  
 3   Margem_Liquida             1000 non-null   float64
 4   Endividamento              1000 non-null   float64
 5   Setor                      1000 non-null   str    
 6   Regiao                     1000 non-null   str    
 7   Tempo_Operacao             1000 non-null   int64  
 8   Auditoria_Externa          1000 non-null   int64  
 9   Rating_Credito             1000 non-null   float64
 10  Tipo_Empresa               1000 non-null   str    
 11  Politica_Sustentabilidade  1000 non-null   str    
 12  Estrategia_Expansao        1000 non-null   str    
 13  Gestao_Risco               1000 non-null   str    
 14  Cobe

In [146]:
df_companies.head(10)

,ID,Nome_Empresa,Receita_Anual,Margem_Liquida,Endividamento,Setor,Regiao,Tempo_Operacao,Auditoria_Externa,Rating_Credito,...,Estrategia_Expansao,Gestao_Risco,Cobertura_Seguros,Maturidade_Digital,Governanca_Corporativa,Cultura_Inovacao,Relacao_Comunidade,Risco_Credito,Risco_Compliance,Risco_Mercado
0,1,Hahn Group,6523388,0.482879,1.032767,Manufatura,Europa,26,0,0.938715,...,Parcerias,Centralizada,Básica,Avançada,Fraca,Neutra,Regular,0,0,0
1,2,Lopez Group,6650634,0.121292,0.492841,Tecnologia,Europa,20,1,0.492362,...,Orgânica,Centralizada,Básica,Inicial,Média,Neutra,Ruim,0,0,0
2,3,Sparks and Sons,4404572,0.190114,0.757099,Manufatura,América Latina,6,0,0.700866,...,Orgânica,Decentralizada,Básica,Inicial,Alta,Neutra,Boa,0,0,0
3,4,"Fields, Ramirez and Craig",2334489,0.402442,2.327962,Manufatura,Europa,6,1,0.855551,...,Parcerias,Centralizada,Nenhuma,Intermediária,Média,Inovadora,Excelente,1,0,0
4,5,"Campbell, Hernandez and Lyons",9624682,0.174549,1.722357,Saúde,América do Norte,18,1,0.418291,...,Aquisições,Decentralizada,Nenhuma,Avançada,Fraca,Neutra,Regular,0,0,0
5,6,Robinson Ltd,7304212,-0.070876,1.290515,Saúde,América do Norte,4,1,0.309413,...,Orgânica,Centralizada,Ampla,Avançada,Alta,Neutra,Ruim,1,0,1
6,7,Bennett LLC,9728519,0.009717,0.130516,Tecnologia,Ásia,17,0,0.936725,...,Parcerias,Decentralizada,Básica,Avançada,Média,Neutra,Regular,0,1,0
7,8,"Rios, Stevens and Johnson",4572471,0.016956,2.481797,Saúde,Europa,14,1,0.515677,...,Parcerias,Centralizada,Nenhuma,Intermediária,Alta,Neutra,Regular,1,0,0
8,9,"Murphy, Walters and Cruz",4623669,0.078123,1.711492,Tecnologia,América Latina,14,0,0.497254,...,Aquisições,Decentralizada,Básica,Avançada,Média,Neutra,Ruim,0,1,0
9,10,Larson Ltd,7504852,0.098752,0.860614,Financeiro,América Latina,13,0,0.178416,...,Aquisições,Centralizada,Básica,Intermediária,Média,Conservadora,Boa,1,1,0


## Análise Exploratória

In [147]:
df_eda = df_companies.copy()

### Valores únicos para as variáveis categóricas
- Não faz sentido nesse caso usar o nome da empresa
- Nenhuma coluna assume somente 1 valor.

In [148]:
for col in df_eda.select_dtypes('str').columns.tolist():
  if col == 'Nome_Empresa':
    continue
  print(f'{col} : {list(df_companies[col].unique())}')

Setor : ['Manufatura', 'Tecnologia', 'Saúde', 'Financeiro']
Regiao : ['Europa', 'América Latina', 'América do Norte', 'Ásia']
Tipo_Empresa : ['MEI', 'S.A.', 'Limitada', 'Multinacional']
Politica_Sustentabilidade : ['Baixa', 'Alta', 'Média']
Estrategia_Expansao : ['Parcerias', 'Orgânica', 'Aquisições']
Gestao_Risco : ['Centralizada', 'Decentralizada']
Cobertura_Seguros : ['Básica', 'Nenhuma', 'Ampla']
Maturidade_Digital : ['Avançada', 'Inicial', 'Intermediária']
Governanca_Corporativa : ['Fraca', 'Média', 'Alta']
Cultura_Inovacao : ['Neutra', 'Inovadora', 'Conservadora']
Relacao_Comunidade : ['Regular', 'Ruim', 'Boa', 'Excelente']


### Descrição das Variáveis Numéricas

In [149]:
df_eda.describe()

,ID,Receita_Anual,Margem_Liquida,Endividamento,Tempo_Operacao,Auditoria_Externa,Rating_Credito,Risco_Credito,Risco_Compliance,Risco_Mercado
count,1000.000000,1.000000e+03,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,500.500000,4.992928e+06,0.152379,1.317909,25.367000,0.496000,0.487000,0.595000,0.206000,0.110000
std,288.819436,2.804931e+06,0.199511,0.700616,14.103873,0.500234,0.292846,0.491138,0.404633,0.313046
min,1.000000,1.393530e+05,-0.199834,0.100452,1.000000,0.000000,0.000748,0.000000,0.000000,0.000000
25%,250.750000,2.646178e+06,-0.021089,0.691918,13.000000,0.000000,0.232142,0.000000,0.000000,0.000000
50%,500.500000,5.032603e+06,0.160441,1.364206,25.000000,0.000000,0.475893,1.000000,0.000000,0.000000
75%,750.250000,7.270658e+06,0.323736,1.919788,38.000000,1.000000,0.742092,1.000000,0.000000,0.000000
max,1000.000000,9.989550e+06,0.499547,2.499313,49.000000,1.000000,0.999049,1.000000,1.000000,1.000000


### Distribuição das Features Numéricas
- Algumas variáveis seguem u8ma distribuição praticamente uniforme.
- Nitidamente não seguem uma distribuição normal.

In [150]:
for col in df_eda.select_dtypes(include=['float64', 'int64']).columns.tolist():
  if col == 'ID' or col.startswith('Risco'):
    continue

  fig  = px.histogram(
    df_eda,
    x=col,
    title=f'Distribuição da Feature {col.title()}',
    nbins=20,
  )

  fig.show()

### Distribuição das Variáveis Categóricas
- Os valores tem aproximadamente a mesma frequência nas variáveis em geral.

In [151]:
for col in df_eda.select_dtypes(include='str').columns.tolist():
    if col == 'Nome_Empresa':
        continue

    table = df_eda.value_counts(col).sort_values(ascending=True)

    fig = px.bar(
        table,
        orientation='h',
        color=table.values,
        color_continuous_scale=px.colors.carto.Blugrn
    )

    fig.show()

### Distribuição: Variáveis Target
- 59.5% das empresas do dataset têm Risco de Crédito.
- 20.6% das empresas do dataset têm Risco de Compliance.
- 11.0% das empresas do dataset têm risco de Mercado. 

In [152]:
for col in df_eda.columns.tolist():
    if not col.startswith('Risco'):
        continue

    table = df_eda.value_counts(col, normalize=True)

    fig = px.bar(
      table,
      color=table.values,
      color_continuous_scale=['darkred', 'darkblue'],
      orientation='h',
      labels={ col: col, 'value': "Frequência Percentual (%)"},
      title=f"Distribuição Percentual da Variável {col.title()}",
    )

    fig.show()
    

### Analisar Relação entre Features Categóricas e Target

In [153]:
targets = ['Risco_Compliance', 'Risco_Credito', 'Risco_Mercado']

for target in targets:
  for col in df_eda.select_dtypes('number').columns.tolist():
      if col == 'ID' or col.startswith('Risco'):
          continue

      fig = px.box(
        df_eda,
        x=target,
        y=col,
        title=f'Boxplot: {col} por {target}',
      )

      fig.show()

  for col in df_eda.select_dtypes('str').columns.tolist():
      if col == 'Nome_Empresa':
          continue
     
      fig = px.histogram(
        df_eda,
        x=col,
        color=target,
        title=f'Histograma: {col} por {target}',
        barmode='group',
        orientation='v',
      )
     
      fig.show()

### Análise de Correlação

In [ ]:
corr_matrix = df_eda.select_dtypes('number').drop(columns=['ID']).corr()

fig = px.imshow(
  corr_matrix,
  color_continuous_scale=px.colors.diverging.RdBu_r,
  zmax=1,
  zmin=-1,
  title="Correlação de Pearson"
)

fig.update_traces(text=corr_matrix, texttemplate='%{text:.3f}', textfont=dict(size=12))

fig.update_layout(width=1000, height=600, title_font=dict(size=24), font=dict(size=12))

### Testes de Hipótese

#### T-Student
- Análise de Variância (diferença significativa média entre 2 grupos).

In [ ]:
for col in df_eda.select_dtypes('number').drop(columns=['ID', 'Risco_Compliance', 'Risco_Credito', 'Risco_Mercado']).columns.tolist():
    for target in targets:
        if df_eda[target].nunique() <= 2:
            groups = [df_eda[df_eda[target] == val][col] for val in df_eda[target].unique()]

            stat, p_value = ttest_ind(groups[0], groups[1])

            print(f'{Fore.RED if p_value < 0.05 else Fore.WHITE}' f'T-Test entre {col} e {target}: p-valor = {p_value}')

T-Test entre Receita_Anual e Risco_Compliance: p-valor = 0.3724539073163081
T-Test entre Receita_Anual e Risco_Credito: p-valor = 0.9148230907469924
T-Test entre Receita_Anual e Risco_Mercado: p-valor = 1.432266289916743e-06
T-Test entre Margem_Liquida e Risco_Compliance: p-valor = 7.293209908091e-67
T-Test entre Margem_Liquida e Risco_Credito: p-valor = 0.714839504289611
T-Test entre Margem_Liquida e Risco_Mercado: p-valor = 0.46787005387671987
T-Test entre Endividamento e Risco_Compliance: p-valor = 0.02873944772709319
T-Test entre Endividamento e Risco_Credito: p-valor = 2.0286202656673827e-48
T-Test entre Endividamento e Risco_Mercado: p-valor = 0.6427051699118196
T-Test entre Tempo_Operacao e Risco_Compliance: p-valor = 0.6875261988254968
T-Test entre Tempo_Operacao e Risco_Credito: p-valor = 0.7368086294717285
T-Test entre Tempo_Operacao e Risco_Mercado: p-valor = 9.253489005940342e-41
T-Test entre Auditoria_Externa e Risco_Compliance: p-valor = 6.428459257234918e-66
T-Test entre

#### ANOVA
- Diferença significativa média entre 3 ou mais grupos.
- Não há casos para 3 ou mais grupos, pois todas targets são binárias.

In [157]:
for col in df_eda.select_dtypes('number').drop(columns=['ID', 'Risco_Compliance', 'Risco_Credito', 'Risco_Mercado']).columns.tolist():
    for target in targets:
        if df_eda[target].nunique() > 2:
            groups = [df_eda[df_eda[target] == val][col] for val in df_eda[target].unique()]

            stat, p_value = f_oneway(*groups)

            print(f'{Fore.RED if p_value < 0.05 else Fore.WHITE}' f'T-Test entre {col} e {target}: p-valor = {p_value}')

## Treinamento do Modelo One vs All